In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import os
import time
import shap
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
warnings.filterwarnings('ignore')
# =====================================
# device
# =====================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[{time.strftime('%H:%M:%S')}] Using device: {device}")
# =====================================
# MLP模型
# =====================================
class RegressionModel(nn.Module):
    def __init__(self, layers_list):
        super(RegressionModel, self).__init__()
        layers = []
        for i in range(len(layers_list) - 1):
            layers.append(nn.Linear(layers_list[i], layers_list[i + 1]))
            if i < len(layers_list) - 2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)
# =====================================
# DataLoader
# ====================================
def get_loader(X, y, bs):
    dataset = TensorDataset(X, y)
    return DataLoader(
        dataset,
        batch_size=bs,
        shuffle=True,
        num_workers=30,
        pin_memory=True,
        persistent_workers=True
    )

# =====================================
# 主训练函数
# =====================================

def train_unified_model():

    print("\n" + "=" * 60)
    print("🚀 Training Unified Model")
    print("=" * 60)

    # =====================================
    # 读取总数据
    # =====================================
    data_path = '~/private_data/20260918/CMAQ_trainingset_2015-2019_ALL.csv'
    data = pd.read_csv(data_path).dropna()
    print(f"数据量: {len(data)}")
    # ====================================
    # bias
    # =====================================
    data['O3_bias'] = data['O3_true'] - data['O3']
    data['PM25_bias'] = data['PM25_true'] - data['PM25']
    # =====================================
    # 周期特征
    # =====================================
    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
    data['time'] = pd.to_datetime(data['time'])
    doy = data['time'].dt.dayofyear
    data['doy_sin'] = np.sin(2 * np.pi * doy / 365)
    data['doy_cos'] = np.cos(2 * np.pi * doy / 365)
    data['POA_EC_ratio'] = data['POA'] / data['EC']
    # =====================================
    # 特征列
    # =====================================
    exclude_cols = [
        'time',
        'O3_true',
        'PM25_true',
        'O3_bias',
        'PM25_bias'
    ]
    feature_cols = [c for c in data.columns if c not in exclude_cols]
    print("\nFeatures:")
    print(feature_cols)
    # =====================================
    # train val split
    # =====================================
    train_df, val_df = train_test_split(data, test_size=0.2, random_state=42)
    # =====================================
    # scaler
    # =====================================
    scaler = StandardScaler()
    # =====================================
    # X y
    # =====================================
    X_train_np = scaler.fit_transform(train_df[feature_cols].values).astype(np.float32)
    X_val_np = scaler.transform(val_df[feature_cols].values).astype(np.float32)
    y_train_np = train_df[['O3_bias', 'PM25_bias']].values.astype(np.float32)
    y_val_np = val_df[['O3_bias', 'PM25_bias']].values.astype(np.float32)
    X_train = torch.tensor(X_train_np, dtype=torch.float32)
    y_train = torch.tensor(y_train_np, dtype=torch.float32)
    X_val = torch.tensor(X_val_np, dtype=torch.float32).to(device)
    y_val = torch.tensor(y_val_np, dtype=torch.float32).to(device)
    # =====================================
    # 超参数
    # =====================================
    input_dim = len(feature_cols)
    param_grid = {
        'hidden_configs': [
            [input_dim, 2048, 1024, 512, 256, 2],
            [input_dim, 1024, 512, 256, 128, 2],
            [input_dim, 512, 256, 128, 64, 2],
        ],
        'lrs': [0.01, 0.001],
        'batch_sizes': [512, 1024, 2048]
    }
    print("\n🔎 Searching hyperparameters...")
    best_val_loss = float('inf')
    best_cfg = {}
    configs = [
        (h, lr, bs)
        for h in param_grid['hidden_configs']
        for lr in param_grid['lrs']
        for bs in param_grid['batch_sizes']
    ]

    # =====================================
    # 超参数搜索
    # =====================================
    for h_cfg, lr, bs in configs:
        print(f"\nTesting: h={h_cfg}, lr={lr}, bs={bs}")
        train_loader = get_loader(X_train, y_train, bs)
        model = RegressionModel(h_cfg).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.MSELoss()
        for epoch in range(20):
            model.train()
            for xb, yb in train_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val), y_val).item()
        print(f"Val Loss: {val_loss:.6f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_cfg = {
                'h': h_cfg,
                'lr': lr,
                'bs': bs
            }
        del model
        del optimizer
        torch.cuda.empty_cache()
    print("\n✅ Best Config:")
    print(best_cfg)

    # =====================================
    # 正式训练
    # =====================================
    train_loader = get_loader(X_train, y_train, best_cfg['bs'])
    model = RegressionModel(best_cfg['h']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=best_cfg['lr'])
    criterion = nn.MSELoss()
    best_wts = None
    min_val_loss = float('inf')
    counter = 0
    print("\n🚀 Final Training...")
    for epoch in range(300):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        # validation
        model.eval()
        with torch.no_grad():
            curr_val_loss = criterion(model(X_val), y_val).item()
        # early stop
        if curr_val_loss < min_val_loss:
            min_val_loss = curr_val_loss
            counter = 0
            best_wts = copy.deepcopy(model.state_dict())
        else:
            counter += 1
        if (epoch + 1) % 20 == 0:
            print(f"Epoch [{epoch+1}/300] | Val Loss: {curr_val_loss:.6f}")
        if counter >= 30:
            print(f"\n⏹ Early stop at epoch {epoch+1}")
            break
    # =====================================
    # load best model
    # =====================================
    model.load_state_dict(best_wts)
    model.eval()
    # =====================================
    # 保存模型
    # =====================================
    save_dir = 'CMAQ_MLP_Model_32features'
    os.makedirs(save_dir, exist_ok=True)
    model_path = f'{save_dir}/unified_model.pth'
    torch.save({
        'model_state_dict': best_wts,
        'scaler': scaler,
        'h_layers': best_cfg['h'],
        'feature_cols': feature_cols,
        'best_params': best_cfg,
        'final_val_loss': min_val_loss
    }, model_path)
    print(f"\n💾 Saved model:")
    print(model_path)
    # =====================================
    # SHAP
    # =====================================
    print("\n🔍 Calculating SHAP values...")
    background_size = 10000
    explain_size = 50000
    background_idx = np.random.choice(len(X_train_np), background_size, replace=False)
    explain_idx = np.random.choice(len(X_val_np), explain_size, replace=False)
    background_data = torch.tensor(X_train_np[background_idx], dtype=torch.float32).to(device)
    explain_data = torch.tensor(X_val_np[explain_idx], dtype=torch.float32).to(device)
    explainer = shap.DeepExplainer(model, background_data)
    shap_values = explainer.shap_values(explain_data)

    # =====================================
    # save SHAP values
    # =====================================
    shap_dir = f'{save_dir}/SHAP'
    os.makedirs(shap_dir, exist_ok=True)
    o3_shap_df = pd.DataFrame(shap_values[0], columns=feature_cols)
    o3_shap_df.to_csv(f'{shap_dir}/O3_shap_values.csv', index=False)
    pm25_shap_df = pd.DataFrame(shap_values[1], columns=feature_cols)
    pm25_shap_df.to_csv(f'{shap_dir}/PM25_shap_values.csv', index=False)
    explain_df = pd.DataFrame(explain_data.cpu().numpy(), columns=feature_cols)
    explain_df.to_csv(f'{shap_dir}/explain_data.csv', index=False)
    pd.DataFrame({
        'feature': feature_cols
    }).to_csv(f'{shap_dir}/feature_names.csv', index=False)
    print("\n💾 SHAP saved:")
    print(shap_dir)
    print("\n✅ Finished!")
# =====================================
# main
# =====================================
if __name__ == "__main__":
    train_unified_model()